In [ ]:
# Install LangChain, Google's integration, and NumPy for vector math
!pip install -q -U langchain-google-genai numpy

In [2]:
import numpy as np
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

import os
load_dotenv()

api_key=os.getenv('GEMINI_API_KEY')

In [3]:


# Initialize the Embedding Model (NOT a chat model!)
embedder = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

print("[System] 768-Dimensional Google Embedding Model Initialized Successfully.")

[System] 768-Dimensional Google Embedding Model Initialized Successfully.


In [4]:
# Define our text data
s1 = "Tony and Marten are scaling the data pipeline to handle more corporate PDFs."
s2 = "The engineering team is upgrading the document ingestion architecture to increase throughput."
s3 = s1 + " " + s1 + " " + s1 + " " + s1 + " " + s1  # Artificial length inflation
s4 = "The client was incredibly happy with the new chocolate cake recipe."
s5 = "Tony and Marten are NOT scaling the data pipeline."

print("Translating human text into mathematical vectors via API...\n")

# Convert text to vectors
v1 = np.array(embedder.embed_query(s1))
v2 = np.array(embedder.embed_query(s2))
v3 = np.array(embedder.embed_query(s3))
v4 = np.array(embedder.embed_query(s4))
v5 = np.array(embedder.embed_query(s5))

# Inspect the raw math
print(f"Total Dimensions per vector: {len(v1)}")
print(f"Sample of Vector 1 (first 5 dims): {v1[:5]}\n")

Translating human text into mathematical vectors via API...

Total Dimensions per vector: 3072
Sample of Vector 1 (first 5 dims): [ 0.00476996  0.021538    0.00370016 -0.09430175 -0.01562046]



The entire sentence s1 is represented by one 3072-dimensional vector.

If you separately did:

embedder.embed_query("Tony")

then you would also get a vector with 3072 numbers:

Tony
 ↓
[0.00..., 0.02..., -0.01..., ..., 0.04...]

In [5]:
def calc_cosine(vec_a, vec_b):
    """Calculates Cosine Similarity (Angle). Returns -1.0 to +1.0."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot_product / (norm_a * norm_b)

def calc_euclidean(vec_a, vec_b):
    """Calculates Euclidean Distance (Straight Line). Returns 0 to Infinity (lower is closer)."""
    return np.linalg.norm(vec_a - vec_b)

print("="*60)
print(" SEMANTIC SEARCH ALGEBRA LABORATORY ")
print("="*60 + "\n")

# ---------------------------------------------------------
# TEST A: SYNONYM UNDERSTANDING
# ---------------------------------------------------------
print("TEST A: Pipeline (v1) vs Engineering Upgrades (v2)")
print(f"  Cosine Similarity: {calc_cosine(v1, v2):.4f}  <-- High score! Meaning is captured.")
print(f"  Euclidean Dist:    {calc_euclidean(v1, v2):.4f}\n")

# ---------------------------------------------------------
# TEST B: THE LENGTH ILLUSION (MAGNITUDE)
# ---------------------------------------------------------
print("TEST B: Pipeline (v1) vs Long Pipeline Document (v3)")
print(f"  Cosine Similarity: {calc_cosine(v1, v3):.4f}  <-- Perfect! It ignores the length.")
print(f"  Euclidean Dist:    {calc_euclidean(v1, v3):.4f}  <-- FAILS! The distance is massive because v3 is so long.\n")

# ---------------------------------------------------------
# TEST C: UNRELATED NOISE
# ---------------------------------------------------------
print("TEST C: Pipeline (v1) vs Chocolate Cake (v4)")
print(f"  Cosine Similarity: {calc_cosine(v1, v4):.4f}  <-- Low score. Mathematically unrelated.")
print(f"  Euclidean Dist:    {calc_euclidean(v1, v4):.4f}\n")

# ---------------------------------------------------------
# TEST D: THE NEGATION EDGE CASE
# ---------------------------------------------------------
print("TEST D: Pipeline (v1) vs NOT Pipeline (v5)")
print(f"  Cosine Similarity: {calc_cosine(v1, v5):.4f}")
print("  -> ANALYSIS: Notice this score! It is very high, even though they mean opposite things.")
print("  -> WHY? Because both sentences are heavily discussing 'Tony', 'Marten', and 'Pipelines'.")
print("  -> In high-dimensional space, they are in the exact same 'neighborhood'.")
print("  -> Takeaway: Basic Cosine search struggles with negation. This is why LLMs are needed to read the chunks later!")
print("="*60)

 SEMANTIC SEARCH ALGEBRA LABORATORY 

TEST A: Pipeline (v1) vs Engineering Upgrades (v2)
  Cosine Similarity: 0.7411  <-- High score! Meaning is captured.
  Euclidean Dist:    0.7196

TEST B: Pipeline (v1) vs Long Pipeline Document (v3)
  Cosine Similarity: 0.9533  <-- Perfect! It ignores the length.
  Euclidean Dist:    0.3057  <-- FAILS! The distance is massive because v3 is so long.

TEST C: Pipeline (v1) vs Chocolate Cake (v4)
  Cosine Similarity: 0.5429  <-- Low score. Mathematically unrelated.
  Euclidean Dist:    0.9561

TEST D: Pipeline (v1) vs NOT Pipeline (v5)
  Cosine Similarity: 0.8778
  -> ANALYSIS: Notice this score! It is very high, even though they mean opposite things.
  -> WHY? Because both sentences are heavily discussing 'Tony', 'Marten', and 'Pipelines'.
  -> In high-dimensional space, they are in the exact same 'neighborhood'.
  -> Takeaway: Basic Cosine search struggles with negation. This is why LLMs are needed to read the chunks later!
